# Submit IGVF file-format specification Documents and attach them to analysis files

This notebook performs the workflow requested by Ian:

1. locate the four PDF file-format specifications under
   `$OAK/Users/tri/tmp/igvf_submit_validation`;
2. POST each PDF as an IGVF `Document` with
   `document_type = file format specification`;
3. retrieve and verify the four Document records;
4. PATCH the corresponding `TabularFile` records through
   `file_format_specifications`;
5. export a final reconciliation table with Document IDs, target-file
   accessions, Portal links, and verification status.

The notebook is restart-safe:

- an existing matching Document is skipped;
- an analysis file already linked to the correct Document is skipped;
- a target file that does not yet exist is reported as `WAITING_FOR_FILE`;
- production work is gated behind separate confirmation tokens;
- each POST/PATCH uses a one-row TSV and a separate log.

The patch operation **extends** the existing
`file_format_specifications` array; it does not overwrite unrelated
Document links.

## Version 2 — submit Document payloads as JSON, not TSV

The first version serialized the nested `attachment` object through
`pandas.to_csv()`. That produced CSV-style doubled quotes inside the
TSV field, for example:

```text
"{""path"":""/oak/.../specification.pdf""}"
```

`iu_register.py` reads TSV rows by splitting on tab characters and
then calls `json.loads()` directly on object-valued fields. It does
not undo CSV quote escaping, so the `attachment` value failed with
`JSONDecodeError: Extra data`.

This version submits each Document from a native `.json` file. The
attachment is therefore represented as the actual JSON object:

```json
{"path": "/oak/.../specification.pdf"}
```

TabularFile PATCH operations remain TSV because they do not contain
nested JSON objects.

## 1. Configuration

In [ ]:
from __future__ import annotations

import gzip
import hashlib
import json
import os
import re
import subprocess
import time
from getpass import getpass
from pathlib import Path
from typing import Any, Sequence

import pandas as pd
import requests

# Sherlock directory supplied by the user.
OAK_ROOT = Path(
    os.environ.get(
        "OAK",
        "/oak/stanford/groups/engreitz",
    )
)
BASE_DIR = OAK_ROOT / "Users/tri/tmp/igvf_submit_validation"
WORK_DIR = BASE_DIR / "_igvf_document_submission"
LOG_DIR = WORK_DIR / "logs"
ROW_DIR = WORK_DIR / "one_row_tsv"

for directory in (WORK_DIR, LOG_DIR, ROW_DIR):
    directory.mkdir(parents=True, exist_ok=True)

AWARD = "/awards/HG011972/"
LAB = "/labs/jesse-engreitz/"

IGVF_PYTHON = Path(
    "/oak/stanford/groups/engreitz/Users/tri/"
    "envs/igvf_utils/bin/python"
)
IU_REGISTER = Path(
    "/oak/stanford/groups/engreitz/Users/tri/"
    "envs/igvf_utils/igvf_utils/igvf_utils/"
    "MetaDataRegistration/iu_register.py"
)

# The four analysis files that should receive the format Documents.
# Replace an alias with an IGVFFI accession when one is known.
SPECIFICATIONS = {
    "validation_guide": {
        "pdf_name": (
            "WTC11_EC_validation_screen_"
            "guide_quantifications_file_format_specification.pdf"
        ),
        "document_alias": (
            "jesse-engreitz:document_WTC11-EC_validation_screen_"
            "guide_quantifications_file_format_v1"
        ),
        "description": (
            "File-format specification for guide-level read-count "
            "quantifications from the focused WTC11-EC validation CRISPRi "
            "screen."
        ),
        "target_file_identifier": "IGVFFI5157STFU",
        "target_file_alias": (
            "jesse-engreitz:tabular_file_WTC11-EC_validation_screen_"
            "guide_quantifications"
        ),
    },
    "validation_differential": {
        "pdf_name": (
            "WTC11_EC_validation_screen_"
            "differential_element_quantifications_"
            "file_format_specification.pdf"
        ),
        "document_alias": (
            "jesse-engreitz:document_WTC11-EC_validation_screen_"
            "differential_element_quantifications_file_format_v1"
        ),
        "description": (
            "File-format specification for promoter-element MAGeCK "
            "differential quantifications from the focused WTC11-EC "
            "validation CRISPRi screen."
        ),
        "target_file_identifier": (
            "jesse-engreitz:tabular_file_WTC11-EC_validation_screen_"
            "D3_CDH5pos_vs_CDH5neg_"
            "differential_element_quantifications"
        ),
        "target_file_alias": (
            "jesse-engreitz:tabular_file_WTC11-EC_validation_screen_"
            "D3_CDH5pos_vs_CDH5neg_"
            "differential_element_quantifications"
        ),
    },
    "genome_wide_guide": {
        "pdf_name": (
            "WTC11_EC_genome_wide_screen_"
            "guide_quantifications_file_format_specification.pdf"
        ),
        "document_alias": (
            "jesse-engreitz:document_WTC11-EC_genome_wide_screen_"
            "guide_quantifications_file_format_v1"
        ),
        "description": (
            "File-format specification for guide-level read-count "
            "quantifications from the genome-wide WTC11-EC CRISPRi screen."
        ),
        "target_file_identifier": (
            "jesse-engreitz:tabular_file_WTC11-EC_genome_wide_screen_"
            "guide_quantifications"
        ),
        "target_file_alias": (
            "jesse-engreitz:tabular_file_WTC11-EC_genome_wide_screen_"
            "guide_quantifications"
        ),
    },
    "genome_wide_differential": {
        "pdf_name": (
            "WTC11_EC_genome_wide_screen_"
            "differential_element_quantifications_"
            "file_format_specification.pdf"
        ),
        "document_alias": (
            "jesse-engreitz:document_WTC11-EC_genome_wide_screen_"
            "differential_element_quantifications_file_format_v1"
        ),
        "description": (
            "File-format specification for native element-level MAGeCK "
            "differential quantifications from the genome-wide WTC11-EC "
            "CRISPRi screen."
        ),
        "target_file_identifier": (
            "jesse-engreitz:tabular_file_WTC11-EC_genome_wide_screen_"
            "D3_CDH5pos_vs_CDH5neg_"
            "differential_element_quantifications"
        ),
        "target_file_alias": (
            "jesse-engreitz:tabular_file_WTC11-EC_genome_wide_screen_"
            "D3_CDH5pos_vs_CDH5neg_"
            "differential_element_quantifications"
        ),
    },
}

print("Base directory:", BASE_DIR)
print("Work directory:", WORK_DIR)
print("igvf_utils Python:", IGVF_PYTHON)
print("iu_register.py:", IU_REGISTER)

## 2. Utility functions

In [ ]:
def write_tsv(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(
        path,
        sep="\t",
        index=False,
        lineterminator="\n",
    )


def md5sum(
    path: Path,
    block_size: int = 16 * 1024 * 1024,
) -> str:
    digest = hashlib.md5()
    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(block_size),
            b"",
        ):
            digest.update(block)
    return digest.hexdigest()


def sha256sum(
    path: Path,
    block_size: int = 16 * 1024 * 1024,
) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(block_size),
            b"",
        ):
            digest.update(block)
    return digest.hexdigest()


def safe_stem(value: str) -> str:
    return re.sub(
        r"[^A-Za-z0-9._-]+",
        "_",
        str(value),
    ).strip("_")


def split_link_items(value: object) -> list[object]:
    if value is None:
        return []
    if isinstance(value, str):
        return [
            item.strip()
            for item in value.split(",")
            if item.strip()
        ]
    if isinstance(value, (list, tuple, set)):
        return list(value)
    return [value]


def terminal_identifier(value: object) -> str:
    if isinstance(value, dict):
        value = (
            value.get("accession")
            or value.get("@id")
            or value.get("uuid")
            or ""
        )
    text = str(value or "").strip().strip("/")
    return text.rsplit("/", 1)[-1] if text else ""


def direct_link_tokens(value: object) -> set[str]:
    tokens: set[str] = set()
    for item in split_link_items(value):
        if isinstance(item, dict):
            for key in ("accession", "uuid", "@id"):
                token = terminal_identifier(
                    item.get(key, "")
                )
                if token:
                    tokens.add(token)
            for alias in split_link_items(
                item.get("aliases", [])
            ):
                alias_text = str(alias).strip()
                if alias_text:
                    tokens.add(alias_text)
        else:
            token = terminal_identifier(item)
            if token:
                tokens.add(token)
            text = str(item).strip()
            if ":" in text and not text.startswith("/"):
                tokens.add(text)
    return tokens


def portal_urls(identifier: str) -> list[str]:
    token = str(identifier).strip().strip("/")
    base = f"https://api.data.igvf.org/{token}/"
    return [
        base + "?format=json&datastore=database",
        base + "?format=json&datastore=database&frame=object",
        base + "?format=json&datastore=database&frame=edit",
    ]


def portal_get(
    identifier: str,
    *,
    timeout: int = 45,
) -> dict | None:
    auth = None
    if (
        os.environ.get("IGVF_API_KEY")
        and os.environ.get("IGVF_SECRET_KEY")
    ):
        auth = (
            os.environ["IGVF_API_KEY"],
            os.environ["IGVF_SECRET_KEY"],
        )

    errors: list[str] = []
    for url in portal_urls(identifier):
        response = requests.get(
            url,
            auth=auth,
            headers={"Accept": "application/json"},
            timeout=timeout,
        )
        if response.status_code == 404:
            continue
        if response.ok:
            return response.json()
        errors.append(
            f"{response.status_code} {url}: "
            f"{response.text[:300]}"
        )

    if errors:
        raise RuntimeError(
            "Portal GET failed:\n" + "\n".join(errors)
        )
    return None


def get_with_retry(
    identifier: str,
    *,
    attempts: int = 6,
) -> dict | None:
    last_error: Exception | None = None
    for attempt in range(1, attempts + 1):
        try:
            record = portal_get(identifier)
            if record is not None:
                return record
        except Exception as error:
            last_error = error

        if attempt < attempts:
            time.sleep(min(2 ** (attempt - 1), 10))

    if last_error is not None:
        raise last_error
    return None


_IDENTITY_CACHE: dict[str, set[str]] = {}


def resolved_link_tokens(value: object) -> set[str]:
    direct = direct_link_tokens(value)
    resolved = set(direct)

    for token in list(direct):
        if token in _IDENTITY_CACHE:
            resolved.update(_IDENTITY_CACHE[token])
            continue

        identities = {token}
        try:
            record = portal_get(token)
        except Exception:
            record = None

        if record:
            identities.update(
                direct_link_tokens(record)
            )

        _IDENTITY_CACHE[token] = identities
        resolved.update(identities)

    return resolved


def links_equal(
    left: object,
    right: object,
) -> bool:
    left_tokens = resolved_link_tokens(left)
    right_tokens = resolved_link_tokens(right)
    return bool(
        left_tokens
        and right_tokens
        and left_tokens & right_tokens
    )


def link_set_contains(
    container: object,
    required: object,
) -> bool:
    existing_items = split_link_items(container)
    required_items = split_link_items(required)
    return all(
        any(
            links_equal(existing, wanted)
            for existing in existing_items
        )
        for wanted in required_items
    )


def portal_record_id(record: dict | None) -> str:
    record = record or {}
    return str(
        record.get("@id")
        or record.get("accession")
        or record.get("uuid")
        or ""
    )


def portal_record_url(record: dict | None) -> str:
    record = record or {}
    portal_id = str(record.get("@id") or "")
    if portal_id:
        return "https://data.igvf.org" + portal_id

    accession = str(record.get("accession") or "")
    if accession:
        return (
            "https://data.igvf.org/"
            f"tabular-files/{accession}/"
        )
    return ""


def row_without_empty_values(row: dict) -> dict:
    return {
        key: value
        for key, value in row.items()
        if str(value).strip() != ""
    }


def run_iu_register(
    *,
    profile: str,
    input_tsv: Path,
    dry_run: bool,
    patch: bool,
    label: str,
) -> dict:
    if not IGVF_PYTHON.is_file():
        raise FileNotFoundError(IGVF_PYTHON)
    if not IU_REGISTER.is_file():
        raise FileNotFoundError(IU_REGISTER)
    if not input_tsv.is_file():
        raise FileNotFoundError(input_tsv)
    if not (
        os.environ.get("IGVF_API_KEY")
        and os.environ.get("IGVF_SECRET_KEY")
    ):
        raise RuntimeError(
            "IGVF credentials are not set"
        )

    timestamp = time.strftime("%Y%m%d_%H%M%S")
    mode = "dryrun" if dry_run else "submit"
    log_path = (
        LOG_DIR
        / f"{safe_stem(label)}_{mode}_{timestamp}.log"
    )

    command = [
        str(IGVF_PYTHON),
        str(IU_REGISTER),
        "-m",
        "prod",
        "-p",
        profile,
        "--tries",
        "3",
        "--delay",
        "10",
        "--backoff",
        "2",
    ]
    if dry_run:
        command.append("-d")
    if patch:
        command.append("--patch")
    command.extend(["-i", str(input_tsv)])

    print("\nCommand:", " ".join(command))
    print("Log:", log_path)

    completed = subprocess.run(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
        check=False,
    )
    log_path.write_text(
        completed.stdout,
        encoding="utf-8",
    )
    print(completed.stdout)

    fatal_patterns = [
        r"Traceback",
        r"ValidationFailure",
        r"Failed to POST",
        r"Failed to PATCH",
        r"\b401 Client Error\b",
        r"\b403 Client Error\b",
        r"\b409 Client Error\b",
        r"\b422 Client Error\b",
    ]
    detected = [
        pattern
        for pattern in fatal_patterns
        if re.search(
            pattern,
            completed.stdout,
            flags=re.IGNORECASE,
        )
    ]

    success = (
        completed.returncode == 0
        and not detected
    )
    if not success:
        raise RuntimeError(
            f"igvf_utils failed; review {log_path}"
        )

    return {
        "profile": profile,
        "input_tsv": str(input_tsv),
        "dry_run": dry_run,
        "patch": patch,
        "log_path": str(log_path),
        "success": success,
    }


print("Utility functions loaded.")

## 3. Validate the four local PDF attachments and build native JSON Document payloads

In [ ]:
if not BASE_DIR.is_dir():
    raise FileNotFoundError(
        f"Submission directory does not exist: {BASE_DIR}"
    )

inventory_rows: list[dict] = []
document_payloads: list[dict] = []

for key, spec in SPECIFICATIONS.items():
    pdf_path = BASE_DIR / spec["pdf_name"]

    if not pdf_path.is_file():
        raise FileNotFoundError(
            f"Missing PDF for {key}: {pdf_path}"
        )
    if pdf_path.stat().st_size == 0:
        raise RuntimeError(
            f"PDF is empty: {pdf_path}"
        )

    with pdf_path.open("rb") as handle:
        header = handle.read(5)
    if header != b"%PDF-":
        raise RuntimeError(
            f"File does not have a PDF header: {pdf_path}"
        )

    inventory_rows.append(
        {
            "key": key,
            "pdf_path": str(pdf_path),
            "size_bytes": pdf_path.stat().st_size,
            "md5sum": md5sum(pdf_path),
            "sha256": sha256sum(pdf_path),
            "document_alias": spec["document_alias"],
            "target_file_identifier": (
                spec["target_file_identifier"]
            ),
        }
    )

    # JSON input must use schema-native types:
    # - aliases is an array
    # - attachment is an object, not a JSON string embedded in TSV
    payload = {
        "aliases": [spec["document_alias"]],
        "document_type": "file format specification",
        "description": spec["description"],
        "attachment": {"path": str(pdf_path)},
        "lab": LAB,
        "award": AWARD,
    }

    # Immediate JSON round-trip validation.
    payload = json.loads(
        json.dumps(
            payload,
            ensure_ascii=False,
            separators=(",", ":"),
        )
    )
    if payload["attachment"] != {"path": str(pdf_path)}:
        raise RuntimeError(
            f"Attachment JSON round-trip failed for {key}"
        )

    document_payloads.append(payload)

pdf_inventory = pd.DataFrame(inventory_rows)
PDF_INVENTORY_PATH = (
    WORK_DIR / "pdf_attachment_inventory.tsv"
)
write_tsv(pdf_inventory, PDF_INVENTORY_PATH)

# This JSON file is suitable for direct use with iu_register.py.
DOCUMENT_METADATA_PATH = (
    WORK_DIR / "document_file_format_specifications.json"
)
DOCUMENT_METADATA_PATH.write_text(
    json.dumps(
        document_payloads,
        indent=2,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)

# A flattened TSV is written only for human review. Do not submit it.
review_rows = []
for payload in document_payloads:
    review_rows.append(
        {
            "aliases": ",".join(payload["aliases"]),
            "document_type": payload["document_type"],
            "description": payload["description"],
            "attachment_path": payload["attachment"]["path"],
            "lab": payload["lab"],
            "award": payload["award"],
        }
    )

DOCUMENT_REVIEW_TSV_PATH = (
    WORK_DIR / "document_file_format_specifications_review_only.tsv"
)
write_tsv(
    pd.DataFrame(review_rows),
    DOCUMENT_REVIEW_TSV_PATH,
)

# Validate the aggregate JSON from disk.
loaded_payloads = json.loads(
    DOCUMENT_METADATA_PATH.read_text(encoding="utf-8")
)
if loaded_payloads != document_payloads:
    raise RuntimeError(
        "Document metadata JSON changed during disk round-trip"
    )

print(pdf_inventory.to_string(index=False))
print("\nSubmission JSON:")
print(DOCUMENT_METADATA_PATH)
print(
    json.dumps(
        document_payloads,
        indent=2,
        ensure_ascii=False,
    )
)
print("\nReview-only TSV:")
print(DOCUMENT_REVIEW_TSV_PATH)

## 4. Enter IGVF credentials

Run the first cell to define the secure helpers, then run the second cell to
open the hidden API-key and secret-key prompts. Repeat both cells after a
kernel restart.

In [ ]:
def enter_igvf_credentials() -> None:
    api_key = getpass(
        "IGVF API key: "
    ).strip()
    secret_key = getpass(
        "IGVF secret key: "
    ).strip()

    if not api_key or not secret_key:
        raise ValueError(
            "Both IGVF credentials are required"
        )

    for name, value in {
        "IGVF_API_KEY": api_key,
        "IGVF_SECRET_KEY": secret_key,
    }.items():
        if (
            len(value) >= 2
            and value[0] in {"'", '"'}
            and value[-1] == value[0]
        ):
            raise ValueError(
                f"{name} has surrounding quotes. "
                "Paste the raw value."
            )

    os.environ["IGVF_API_KEY"] = api_key
    os.environ["IGVF_SECRET_KEY"] = secret_key

    del api_key, secret_key
    print(
        "IGVF credentials are set for "
        "the current notebook kernel."
    )


def show_credential_status() -> None:
    for variable in (
        "IGVF_API_KEY",
        "IGVF_SECRET_KEY",
    ):
        value = os.environ.get(variable, "")
        print(
            f"{variable}: "
            f"{'SET' if value else 'MISSING'}, "
            f"length={len(value)}, "
            f"whitespace={value != value.strip()}"
        )

In [ ]:
# Run this cell before Portal preflight.
enter_igvf_credentials()
show_credential_status()

## 5. Build the Document submission plan

Existing matching Document aliases are skipped. A pre-existing alias with a
different `document_type` is treated as a blocking conflict.

In [ ]:
if not (
    os.environ.get("IGVF_API_KEY")
    and os.environ.get("IGVF_SECRET_KEY")
):
    raise RuntimeError(
        "Run the credential-entry cell first"
    )

document_plan_rows: list[dict] = []

for key, spec in SPECIFICATIONS.items():
    alias = spec["document_alias"]

    try:
        record = get_with_retry(alias)
        lookup_error = ""
    except Exception as error:
        record = None
        lookup_error = (
            f"{type(error).__name__}: {error}"
        )

    if lookup_error:
        action = "BLOCK_LOOKUP_ERROR"
    elif record is None:
        action = "POST_NEW"
    elif (
        str(record.get("document_type", ""))
        != "file format specification"
    ):
        action = "BLOCK_EXISTING_DOCUMENT_CONFLICT"
    else:
        action = "SKIP_EXISTING"

    attachment = (
        record.get("attachment", {})
        if record
        else {}
    )

    document_plan_rows.append(
        {
            "key": key,
            "document_alias": alias,
            "action": action,
            "portal_id": str(
                (record or {}).get("@id", "")
            ),
            "portal_uuid": str(
                (record or {}).get("uuid", "")
            ),
            "portal_document_type": str(
                (record or {}).get(
                    "document_type",
                    "",
                )
            ),
            "portal_attachment_download": str(
                attachment.get("download", "")
                if isinstance(attachment, dict)
                else ""
            ),
            "portal_url": portal_record_url(record),
            "lookup_error": lookup_error,
        }
    )

document_plan = pd.DataFrame(
    document_plan_rows
)
DOCUMENT_PLAN_PATH = (
    WORK_DIR / "document_submission_plan.tsv"
)
write_tsv(document_plan, DOCUMENT_PLAN_PATH)

print(document_plan.to_string(index=False))
print("\nSaved:", DOCUMENT_PLAN_PATH)

blocked = document_plan.loc[
    document_plan["action"].str.startswith(
        "BLOCK_"
    )
]
if not blocked.empty:
    raise RuntimeError(
        "The Document plan contains a blocking "
        f"condition. Review {DOCUMENT_PLAN_PATH}"
    )

## 6. Create one-row Document POST JSON files

Each Document is submitted from a native JSON object. The older
`POST_document_*.tsv` files are removed to prevent accidental reuse.

In [ ]:
metadata_by_alias = {
    payload["aliases"][0]: payload
    for payload in document_payloads
}

# Remove unsafe operation TSVs from version 1. They contain nested JSON
# encoded with CSV quote doubling and must not be submitted.
removed_legacy_files = []
for legacy_path in ROW_DIR.glob("POST_document_*.tsv"):
    legacy_path.unlink()
    removed_legacy_files.append(str(legacy_path))

if removed_legacy_files:
    print("Removed legacy Document POST TSVs:")
    for legacy_path in removed_legacy_files:
        print(" ", legacy_path)

document_operation_rows: list[dict] = []

for plan in document_plan.to_dict("records"):
    key = plan["key"]
    action = plan["action"]
    alias = plan["document_alias"]

    if action == "POST_NEW":
        json_path = (
            ROW_DIR
            / f"POST_document_{safe_stem(key)}.json"
        )
        payload = metadata_by_alias[alias]

        json_path.write_text(
            json.dumps(
                payload,
                indent=2,
                ensure_ascii=False,
            )
            + "\n",
            encoding="utf-8",
        )

        # Validate the exact file that iu_register.py will read.
        reloaded = json.loads(
            json_path.read_text(encoding="utf-8")
        )
        if reloaded != payload:
            raise RuntimeError(
                f"JSON round-trip failed for {json_path}"
            )
        if not isinstance(
            reloaded.get("attachment"),
            dict,
        ):
            raise RuntimeError(
                f"attachment is not a JSON object in {json_path}"
            )
        if set(
            reloaded["attachment"]
        ) != {"path"}:
            raise RuntimeError(
                "attachment must contain exactly one key, 'path': "
                f"{json_path}"
            )
    else:
        json_path = None

    document_operation_rows.append(
        {
            "key": key,
            "document_alias": alias,
            "action": action,
            "input_file": (
                str(json_path)
                if json_path is not None
                else ""
            ),
            "input_format": (
                "json"
                if json_path is not None
                else ""
            ),
        }
    )

document_operations = pd.DataFrame(
    document_operation_rows
)
DOCUMENT_OPERATIONS_PATH = (
    WORK_DIR / "document_submission_operations.tsv"
)
write_tsv(
    document_operations,
    DOCUMENT_OPERATIONS_PATH,
)

print(
    document_operations.to_string(index=False)
)

# Display the first operation payload for visual confirmation.
first_json = next(
    (
        Path(value)
        for value in document_operations["input_file"]
        if str(value).strip()
    ),
    None,
)
if first_json is not None:
    print("\nExample native JSON operation file:")
    print(first_json)
    print(first_json.read_text(encoding="utf-8"))

### 6A. Validate the exact JSON operation files

Run this cell before enabling dry runs. It verifies that every pending
Document operation is valid JSON and that `attachment` is a native
one-key object with a readable local PDF path.

In [ ]:
json_preflight_rows = []

for operation in document_operations.to_dict("records"):
    if operation["action"] == "SKIP_EXISTING":
        json_preflight_rows.append(
            {
                "key": operation["key"],
                "action": operation["action"],
                "input_file": "",
                "valid_json": True,
                "attachment_is_object": True,
                "attachment_exists": True,
                "status": "SKIP_EXISTING",
            }
        )
        continue

    input_path = Path(operation["input_file"])
    payload = json.loads(
        input_path.read_text(encoding="utf-8")
    )
    attachment = payload.get("attachment")
    attachment_is_object = (
        isinstance(attachment, dict)
        and set(attachment) == {"path"}
    )
    attachment_path = (
        Path(attachment["path"])
        if attachment_is_object
        else None
    )
    attachment_exists = bool(
        attachment_path
        and attachment_path.is_file()
    )

    json_preflight_rows.append(
        {
            "key": operation["key"],
            "action": operation["action"],
            "input_file": str(input_path),
            "valid_json": True,
            "attachment_is_object": attachment_is_object,
            "attachment_exists": attachment_exists,
            "status": (
                "PASS"
                if attachment_is_object
                and attachment_exists
                else "FAIL"
            ),
        }
    )

json_preflight = pd.DataFrame(
    json_preflight_rows
)
DOCUMENT_JSON_PREFLIGHT_PATH = (
    WORK_DIR / "document_json_preflight.tsv"
)
write_tsv(
    json_preflight,
    DOCUMENT_JSON_PREFLIGHT_PATH,
)

print(json_preflight.to_string(index=False))
print("\nSaved:", DOCUMENT_JSON_PREFLIGHT_PATH)

if not json_preflight["status"].isin(
    {"PASS", "SKIP_EXISTING"}
).all():
    raise RuntimeError(
        "At least one Document JSON operation failed preflight"
    )

## 7. Dry-run the missing Document records

Set `RUN_DOCUMENT_DRY_RUNS = True`, rerun the cell, and review every log.

In [ ]:
RUN_DOCUMENT_DRY_RUNS = False

DOCUMENT_DRY_RUN_MARKER = (
    WORK_DIR / "DOCUMENT_DRY_RUN_PASSED.json"
)

if not RUN_DOCUMENT_DRY_RUNS:
    print(
        "Document dry runs are disabled. "
        "Set RUN_DOCUMENT_DRY_RUNS=True "
        "and rerun this cell."
    )
else:
    results: list[dict] = []

    for operation in (
        document_operations.to_dict("records")
    ):
        if operation["action"] == "SKIP_EXISTING":
            results.append(
                {
                    "key": operation["key"],
                    "action": "SKIP_EXISTING",
                    "success": True,
                    "log_path": "",
                }
            )
            continue

        result = run_iu_register(
            profile="document",
            input_tsv=Path(
                operation["input_file"]
            ),
            dry_run=True,
            patch=False,
            label=(
                f"document_{operation['key']}"
            ),
        )
        results.append(
            {
                "key": operation["key"],
                "action": operation["action"],
                "success": result["success"],
                "log_path": result["log_path"],
            }
        )

    results_frame = pd.DataFrame(results)
    DOCUMENT_DRY_RUN_RESULTS_PATH = (
        WORK_DIR
        / "document_dry_run_results.tsv"
    )
    write_tsv(
        results_frame,
        DOCUMENT_DRY_RUN_RESULTS_PATH,
    )

    if not results_frame["success"].all():
        raise RuntimeError(
            "At least one Document dry run failed"
        )

    marker = {
        "timestamp": time.strftime(
            "%Y-%m-%dT%H:%M:%S%z"
        ),
        "document_metadata_sha256": (
            sha256sum(DOCUMENT_METADATA_PATH)
        ),
        "pdf_sha256": {
            row["key"]: row["sha256"]
            for row in inventory_rows
        },
    }
    DOCUMENT_DRY_RUN_MARKER.write_text(
        json.dumps(marker, indent=2)
        + "\n",
        encoding="utf-8",
    )

    print("All Document dry runs passed.")
    print("Marker:", DOCUMENT_DRY_RUN_MARKER)

## 8. Submit the missing Document records

After reviewing the dry-run logs, enter the confirmation token and rerun
this cell. The notebook verifies all four aliases afterward.

In [ ]:
DOCUMENT_PRODUCTION_CONFIRMATION = ""
REQUIRED_DOCUMENT_CONFIRMATION = (
    "POST_4_FILE_FORMAT_DOCUMENTS_TO_IGVF_PROD"
)

if (
    DOCUMENT_PRODUCTION_CONFIRMATION
    != REQUIRED_DOCUMENT_CONFIRMATION
):
    print("Document production POST is blocked.")
    print("Required confirmation token:")
    print(REQUIRED_DOCUMENT_CONFIRMATION)
else:
    if not DOCUMENT_DRY_RUN_MARKER.is_file():
        raise RuntimeError(
            "Document dry-run marker is missing"
        )

    marker = json.loads(
        DOCUMENT_DRY_RUN_MARKER.read_text(
            encoding="utf-8"
        )
    )
    if (
        marker["document_metadata_sha256"]
        != sha256sum(DOCUMENT_METADATA_PATH)
    ):
        raise RuntimeError(
            "Document metadata changed after dry run"
        )

    for inventory_row in inventory_rows:
        if (
            marker["pdf_sha256"][
                inventory_row["key"]
            ]
            != inventory_row["sha256"]
        ):
            raise RuntimeError(
                "A PDF changed after dry run: "
                f"{inventory_row['pdf_path']}"
            )

    production_rows: list[dict] = []

    for operation in (
        document_operations.to_dict("records")
    ):
        if operation["action"] == "SKIP_EXISTING":
            production_rows.append(
                {
                    "key": operation["key"],
                    "action": "SKIP_EXISTING",
                    "success": True,
                    "log_path": "",
                }
            )
            continue

        result = run_iu_register(
            profile="document",
            input_tsv=Path(
                operation["input_file"]
            ),
            dry_run=False,
            patch=False,
            label=(
                f"document_{operation['key']}"
            ),
        )
        production_rows.append(
            {
                "key": operation["key"],
                "action": operation["action"],
                "success": result["success"],
                "log_path": result["log_path"],
            }
        )

    production_frame = pd.DataFrame(
        production_rows
    )
    write_tsv(
        production_frame,
        WORK_DIR
        / "document_production_results.tsv",
    )

    verification_rows: list[dict] = []

    for key, spec in SPECIFICATIONS.items():
        record = get_with_retry(
            spec["document_alias"],
            attempts=10,
        )
        if record is None:
            raise RuntimeError(
                "Document was not found after POST: "
                f"{spec['document_alias']}"
            )
        if (
            str(record.get("document_type", ""))
            != "file format specification"
        ):
            raise RuntimeError(
                "Unexpected document_type for "
                f"{spec['document_alias']}"
            )

        verification_rows.append(
            {
                "key": key,
                "document_alias": (
                    spec["document_alias"]
                ),
                "document_id": portal_record_id(
                    record
                ),
                "document_uuid": str(
                    record.get("uuid", "")
                ),
                "portal_url": portal_record_url(
                    record
                ),
                "document_type": str(
                    record.get(
                        "document_type",
                        "",
                    )
                ),
                "attachment_download": str(
                    (
                        record.get(
                            "attachment",
                            {},
                        )
                        or {}
                    ).get("download", "")
                ),
                "verification": "PASS",
            }
        )

    document_verification = pd.DataFrame(
        verification_rows
    )
    DOCUMENT_RESULTS_PATH = (
        WORK_DIR
        / "document_portal_records.csv"
    )
    document_verification.to_csv(
        DOCUMENT_RESULTS_PATH,
        index=False,
        lineterminator="\n",
    )

    if len(document_verification) != 4:
        raise RuntimeError(
            "Expected four verified Documents"
        )

    DOCUMENT_COMPLETE_MARKER = (
        WORK_DIR
        / "DOCUMENTS_SUBMITTED_4_OF_4.txt"
    )
    DOCUMENT_COMPLETE_MARKER.write_text(
        "Four file-format specification "
        "Documents exist on the IGVF Portal.\n"
        f"Records: {DOCUMENT_RESULTS_PATH}\n",
        encoding="utf-8",
    )

    print(document_verification.to_string(
        index=False
    ))
    print("\nMarker:", DOCUMENT_COMPLETE_MARKER)

## 9. Refresh Document records and build the target-file patch plan

This read-only cell can be rerun at any time. A target analysis file that
does not yet exist is reported as `WAITING_FOR_FILE` rather than causing the
notebook to fail.

In [ ]:
document_record_rows: list[dict] = []
document_records: dict[str, dict] = {}

for key, spec in SPECIFICATIONS.items():
    record = get_with_retry(
        spec["document_alias"],
        attempts=4,
    )
    if record is None:
        raise RuntimeError(
            "Required format Document not found: "
            f"{spec['document_alias']}"
        )

    document_records[key] = record
    document_record_rows.append(
        {
            "key": key,
            "document_alias": (
                spec["document_alias"]
            ),
            "document_id": portal_record_id(
                record
            ),
            "portal_url": portal_record_url(
                record
            ),
        }
    )

document_records_frame = pd.DataFrame(
    document_record_rows
)
write_tsv(
    document_records_frame,
    WORK_DIR / "document_records_current.tsv",
)

patch_plan_rows: list[dict] = []

for key, spec in SPECIFICATIONS.items():
    lookup_identifier = (
        spec["target_file_identifier"]
        or spec["target_file_alias"]
    )

    try:
        file_record = get_with_retry(
            lookup_identifier,
            attempts=4,
        )
        lookup_error = ""
    except Exception as error:
        file_record = None
        lookup_error = (
            f"{type(error).__name__}: {error}"
        )

    document_record = document_records[key]
    document_id = portal_record_id(
        document_record
    )
    existing_specs = (
        (file_record or {}).get(
            "file_format_specifications",
            [],
        )
    )

    if lookup_error:
        action = "BLOCK_LOOKUP_ERROR"
    elif file_record is None:
        action = "WAITING_FOR_FILE"
    elif link_set_contains(
        existing_specs,
        document_id,
    ):
        action = "SKIP_ALREADY_LINKED"
    else:
        action = "PATCH_FILE"

    patch_plan_rows.append(
        {
            "key": key,
            "target_file_lookup": (
                lookup_identifier
            ),
            "target_file_alias": (
                spec["target_file_alias"]
            ),
            "action": action,
            "target_accession": str(
                (file_record or {}).get(
                    "accession",
                    "",
                )
            ),
            "target_record_id": (
                portal_record_id(file_record)
            ),
            "target_portal_url": (
                portal_record_url(file_record)
            ),
            "document_alias": (
                spec["document_alias"]
            ),
            "document_id": document_id,
            "existing_file_format_specifications": (
                ",".join(
                    sorted(
                        direct_link_tokens(
                            existing_specs
                        )
                    )
                )
            ),
            "lookup_error": lookup_error,
        }
    )

patch_plan = pd.DataFrame(patch_plan_rows)
PATCH_PLAN_PATH = (
    WORK_DIR / "file_format_patch_plan.tsv"
)
write_tsv(patch_plan, PATCH_PLAN_PATH)

print(patch_plan.to_string(index=False))
print("\nSaved:", PATCH_PLAN_PATH)

blocked = patch_plan.loc[
    patch_plan["action"].eq(
        "BLOCK_LOOKUP_ERROR"
    )
]
if not blocked.empty:
    raise RuntimeError(
        "At least one target-file lookup failed. "
        f"Review {PATCH_PLAN_PATH}"
    )

## 10. Create one-row `TabularFile` PATCH TSVs

In [ ]:
patch_operation_rows: list[dict] = []

for plan in patch_plan.to_dict("records"):
    key = plan["key"]
    action = plan["action"]

    if action == "PATCH_FILE":
        record_id = (
            plan["target_accession"]
            or plan["target_record_id"]
            or plan["target_file_lookup"]
        )
        patch_tsv = (
            ROW_DIR
            / f"PATCH_file_format_{safe_stem(key)}.tsv"
        )
        payload = {
            "record_id": record_id,
            "file_format_specifications": (
                plan["document_id"]
            ),
        }
        write_tsv(
            pd.DataFrame([payload]),
            patch_tsv,
        )
    else:
        patch_tsv = None

    patch_operation_rows.append(
        {
            "key": key,
            "action": action,
            "input_tsv": (
                str(patch_tsv)
                if patch_tsv is not None
                else ""
            ),
        }
    )

patch_operations = pd.DataFrame(
    patch_operation_rows
)
PATCH_OPERATIONS_PATH = (
    WORK_DIR / "file_format_patch_operations.tsv"
)
write_tsv(
    patch_operations,
    PATCH_OPERATIONS_PATH,
)

print(patch_operations.to_string(
    index=False
))

## 11. Dry-run available file patches

The default `igvf_utils` PATCH behavior extends array values and removes
duplicates. It does not overwrite other `file_format_specifications`
entries.

In [ ]:
RUN_PATCH_DRY_RUNS = False

PATCH_DRY_RUN_MARKER = (
    WORK_DIR / "PATCH_DRY_RUN_PASSED.json"
)

if not RUN_PATCH_DRY_RUNS:
    print(
        "Patch dry runs are disabled. "
        "Set RUN_PATCH_DRY_RUNS=True "
        "and rerun this cell."
    )
else:
    results: list[dict] = []

    for operation in (
        patch_operations.to_dict("records")
    ):
        if operation["action"] in {
            "SKIP_ALREADY_LINKED",
            "WAITING_FOR_FILE",
        }:
            results.append(
                {
                    "key": operation["key"],
                    "action": operation["action"],
                    "success": True,
                    "log_path": "",
                }
            )
            continue

        result = run_iu_register(
            profile="tabular_file",
            input_tsv=Path(
                operation["input_file"]
            ),
            dry_run=True,
            patch=True,
            label=(
                f"file_format_patch_"
                f"{operation['key']}"
            ),
        )
        results.append(
            {
                "key": operation["key"],
                "action": operation["action"],
                "success": result["success"],
                "log_path": result["log_path"],
            }
        )

    result_frame = pd.DataFrame(results)
    PATCH_DRY_RUN_RESULTS_PATH = (
        WORK_DIR
        / "file_format_patch_dry_run_results.tsv"
    )
    write_tsv(
        result_frame,
        PATCH_DRY_RUN_RESULTS_PATH,
    )

    if not result_frame["success"].all():
        raise RuntimeError(
            "At least one patch dry run failed"
        )

    marker = {
        "timestamp": time.strftime(
            "%Y-%m-%dT%H:%M:%S%z"
        ),
        "patch_plan_sha256": sha256sum(
            PATCH_PLAN_PATH
        ),
        "operation_files": {
            row["key"]: (
                sha256sum(Path(row["input_tsv"]))
                if row["input_tsv"]
                else ""
            )
            for row in patch_operation_rows
        },
    }
    PATCH_DRY_RUN_MARKER.write_text(
        json.dumps(marker, indent=2)
        + "\n",
        encoding="utf-8",
    )

    print("All available patch dry runs passed.")
    print("Marker:", PATCH_DRY_RUN_MARKER)

## 12. Apply the available file patches

`WAITING_FOR_FILE` entries are not errors. Rerun Sections 9-12 after those
analysis files are posted.

In [ ]:
PATCH_PRODUCTION_CONFIRMATION = ""
REQUIRED_PATCH_CONFIRMATION = (
    "ATTACH_4_FORMAT_DOCUMENTS_TO_ANALYSIS_FILES"
)

if (
    PATCH_PRODUCTION_CONFIRMATION
    != REQUIRED_PATCH_CONFIRMATION
):
    print("Production PATCH is blocked.")
    print("Required confirmation token:")
    print(REQUIRED_PATCH_CONFIRMATION)
else:
    if not PATCH_DRY_RUN_MARKER.is_file():
        raise RuntimeError(
            "Patch dry-run marker is missing"
        )

    marker = json.loads(
        PATCH_DRY_RUN_MARKER.read_text(
            encoding="utf-8"
        )
    )
    if (
        marker["patch_plan_sha256"]
        != sha256sum(PATCH_PLAN_PATH)
    ):
        raise RuntimeError(
            "The patch plan changed after dry run"
        )

    production_rows: list[dict] = []

    for operation in (
        patch_operations.to_dict("records")
    ):
        if operation["action"] in {
            "SKIP_ALREADY_LINKED",
            "WAITING_FOR_FILE",
        }:
            production_rows.append(
                {
                    "key": operation["key"],
                    "action": operation["action"],
                    "success": True,
                    "log_path": "",
                }
            )
            continue

        input_tsv = Path(
            operation["input_file"]
        )
        if (
            marker["operation_files"][
                operation["key"]
            ]
            != sha256sum(input_tsv)
        ):
            raise RuntimeError(
                "A PATCH TSV changed after dry run: "
                f"{input_tsv}"
            )

        result = run_iu_register(
            profile="tabular_file",
            input_tsv=input_tsv,
            dry_run=False,
            patch=True,
            label=(
                f"file_format_patch_"
                f"{operation['key']}"
            ),
        )
        production_rows.append(
            {
                "key": operation["key"],
                "action": operation["action"],
                "success": result["success"],
                "log_path": result["log_path"],
            }
        )

    production_frame = pd.DataFrame(
        production_rows
    )
    PATCH_PRODUCTION_RESULTS_PATH = (
        WORK_DIR
        / "file_format_patch_production_results.tsv"
    )
    write_tsv(
        production_frame,
        PATCH_PRODUCTION_RESULTS_PATH,
    )

    print(production_frame.to_string(
        index=False
    ))
    print(
        "\nProduction results:",
        PATCH_PRODUCTION_RESULTS_PATH,
    )

## 13. Final read-only reconciliation

Rerun this cell after any POST or PATCH attempt. It never changes the
Portal.

In [ ]:
final_rows: list[dict] = []

for key, spec in SPECIFICATIONS.items():
    document_record = get_with_retry(
        spec["document_alias"],
        attempts=4,
    )
    target_lookup = (
        spec["target_file_identifier"]
        or spec["target_file_alias"]
    )
    target_record = get_with_retry(
        target_lookup,
        attempts=4,
    )

    document_id = portal_record_id(
        document_record
    )

    if document_record is None:
        status = "DOCUMENT_MISSING"
    elif target_record is None:
        status = "WAITING_FOR_FILE"
    elif link_set_contains(
        target_record.get(
            "file_format_specifications",
            [],
        ),
        document_id,
    ):
        status = "LINKED"
    else:
        status = "PATCH_NEEDED"

    final_rows.append(
        {
            "key": key,
            "document_alias": (
                spec["document_alias"]
            ),
            "document_id": document_id,
            "document_portal_url": (
                portal_record_url(
                    document_record
                )
            ),
            "target_file_lookup": target_lookup,
            "target_accession": str(
                (target_record or {}).get(
                    "accession",
                    "",
                )
            ),
            "target_portal_url": (
                portal_record_url(
                    target_record
                )
            ),
            "portal_file_format_specifications": (
                ",".join(
                    sorted(
                        direct_link_tokens(
                            (target_record or {}).get(
                                "file_format_specifications",
                                [],
                            )
                        )
                    )
                )
            ),
            "status": status,
        }
    )

final_reconciliation = pd.DataFrame(
    final_rows
)
FINAL_RESULTS_PATH = (
    WORK_DIR
    / "file_format_document_reconciliation.csv"
)
final_reconciliation.to_csv(
    FINAL_RESULTS_PATH,
    index=False,
    lineterminator="\n",
)

print(final_reconciliation.to_string(
    index=False
))
print("\nSaved:", FINAL_RESULTS_PATH)

linked_count = int(
    final_reconciliation["status"]
    .eq("LINKED")
    .sum()
)
print(
    f"\nLinked analysis files: "
    f"{linked_count}/4"
)

if linked_count == 4:
    complete_marker = (
        WORK_DIR
        / "FILE_FORMAT_DOCUMENTS_LINKED_4_OF_4.txt"
    )
    complete_marker.write_text(
        "All four analysis files link to their "
        "file-format specification Documents.\n"
        f"Results: {FINAL_RESULTS_PATH}\n",
        encoding="utf-8",
    )
    print("Marker:", complete_marker)

## 14. Output summary

In [ ]:
output_candidates = [
    ("PDF inventory", PDF_INVENTORY_PATH),
    ("Document metadata JSON", DOCUMENT_METADATA_PATH),
        ("Document review-only TSV", DOCUMENT_REVIEW_TSV_PATH),
        ("Document JSON preflight", WORK_DIR / "document_json_preflight.tsv"),
    ("Document plan", DOCUMENT_PLAN_PATH),
    (
        "Document operations",
        DOCUMENT_OPERATIONS_PATH,
    ),
    (
        "Document dry-run marker",
        DOCUMENT_DRY_RUN_MARKER,
    ),
    (
        "Document Portal records",
        WORK_DIR / "document_portal_records.csv",
    ),
    ("File patch plan", PATCH_PLAN_PATH),
    ("File patch operations", PATCH_OPERATIONS_PATH),
    (
        "Patch dry-run marker",
        PATCH_DRY_RUN_MARKER,
    ),
    (
        "Final reconciliation",
        WORK_DIR
        / "file_format_document_reconciliation.csv",
    ),
]

output_summary = pd.DataFrame(
    [
        {
            "output": label,
            "path": str(path),
            "exists": path.is_file(),
            "size_bytes": (
                path.stat().st_size
                if path.is_file()
                else 0
            ),
        }
        for label, path in output_candidates
    ]
)

print(output_summary.to_string(index=False))
print("\nBase directory:", BASE_DIR)
print("Logs:", LOG_DIR)